# Nemotron 3 Nano 4B: evaluate the endpoint

This track evaluates the endpoint created by notebook 3. It then sends the
already-generated answers to Amazon Bedrock for judging.

| Step | What you do |
| --- | --- |
| 1. Setup and switches | Choose live endpoint scoring, live judging, or saved results independently |
| 2. Statistical scoring | Smoke-test one contract, score the held-out data, and inspect metrics |
| 3. Optional performance | Run the existing streaming benchmark and serving-cost calculations |
| 4. Compare baselines | Read the untuned/frontier results and the comparison chart |
| 5. Bedrock judge | Inspect the rubric, submit the same two jobs, and compare score coverage |
| 6. Error analysis and cleanup | Review confusion directions and delete serving resources |

Supporting code is in [evaluation_utils.py](evaluation_utils.py). The standalone
scorer and [inference_performance.py](inference_performance.py) retain their existing
implementations. The switches, models, request settings, baselines and cleanup
behavior are preserved.

<details>
<summary>Reference results and context from the original lesson</summary>

# Lab 1: Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 4: Evaluation

This notebook scores the fine-tuned model on the 123 held-out contracts and compares three
models on the same task:

| model | accuracy | evidence F1 | contradiction correctness | valid JSON |
| ----- | -------- | ----------- | ---------------- | ---------- |
| Nemotron 3 Nano 4B, untuned | 13.6 | 7.8 | 3.8 | 30.1 |
| **the same 4B, fine-tuned** | **87.9** | **76.6** | **70.1** | **100.0** |
| Claude Sonnet 5 | 84.1 | 68.6 | 62.3 | 100.0 |

A 4B model fine-tuned on 423 contracts scores above Sonnet 5 on this task, and it runs on one
A10G.

All three rows ship with the lab as pre-computed results. Part 1 can re-score the middle row
against your own endpoint; it will differ by a few tenths, because requests are greedy but vLLM
batches whatever is in flight.

Two methods:

| Method | What it is good for |
| ------ | ------------------- |
| **Statistical scoring** | exact, reproducible, free. Was the verdict right? Were the cited spans right? |
| **LLM-as-a-Judge** | semantic. Does the cited clause actually support the verdict, even when it is not the span the annotator picked? |

Parts 1 and 2 are the statistical scoring, done by `contractnli_scorer` over all 123 contracts.
Part 3 is the judge, run as a managed Amazon Bedrock evaluation job over a 30-contract slice.

</details>

## 1. Setup and execution choices

In [ ]:
import json
import os
import time

import boto3
from botocore.config import Config
from sagemaker.core.helper.session_helper import Session, get_execution_role

import contractnli as C
import evaluation_utils as E
from config import BASE_MODEL_ID, MODEL_SLUG

ENDPOINT_NAME = E.rname(f"{MODEL_SLUG}-contractnli", "-sft-ep")
SERVED_MODEL_NAME = "nemotron-contractnli"   # SM_VLLM_SERVED_MODEL_NAME in notebook 3

RECORDS = 123
WORKERS = 8

BASELINES = "baselines"          # pre-computed results that ship with the lab

RUN_ENDPOINT_EVAL = False   # score your endpoint over 123 contracts: about 3 minutes
RUN_JUDGE_EVAL = False      # two Bedrock judge jobs over 30 contracts: about 20 minutes

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(
        RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket_name = sess.default_bucket()
region = sess.boto_region_name

runtime = boto3.client("sagemaker-runtime", region_name=region,
                       config=Config(read_timeout=900, retries={"max_attempts": 0}))
print(f"endpoint: {ENDPOINT_NAME}  region: {region}")
print(f"bucket:   {bucket_name}")

### Use pre-computed results, or run the evaluations yourself

Both switches default to `False`, which reads pre-computed results from `baselines/` and submits no inference or judge
jobs. Setup and cleanup still access AWS. Set a switch to `True` to run that evaluation yourself.

| Switch | `False` (default) | `True` | Time when `True` |
| ------ | ----------------- | ------ | ---------------- |
| `RUN_ENDPOINT_EVAL` | reads pre-computed results | scores all 123 contracts against your endpoint | ~3 min |
| `RUN_JUDGE_EVAL` | reads pre-computed results | runs two Bedrock judge jobs over 30 contracts | ~20 min |

The switches are independent. `RUN_JUDGE_EVAL = True` with `RUN_ENDPOINT_EVAL = False` judges the
pre-computed answers, so it needs no endpoint.

Most of the judge's 20 minutes is fixed startup: a job over two contracts takes eight.

In [ ]:
C.ensure_dataset("./data")
EVAL_DOCS, labels = C.load("test")
EVAL_DOCS = EVAL_DOCS[:RECORDS]
label_keys = list(labels)
print(f"{len(EVAL_DOCS)} contracts, {len(label_keys)} checklist items each")
E.check_scorer(EVAL_DOCS[0], label_keys)

### What we measure

`contractnli_scorer.score_record` compares the model's JSON against the annotator's, per
contract:

- **accuracy**: the share of the 17 verdicts that match.
- **evidence F1**: how well the cited span numbers match the annotator's, so a model cannot
  score well by guessing labels without pointing at the text.
- **contradiction correctness**: mean per-contract fraction of gold `Contradiction` items predicted correctly; contracts with no gold contradictions contribute zero, as in the existing scorer. This is not F1; the stored key remains `contradiction_correct`.
- **valid JSON**: the scorer extracted a JSON object, after stripping reasoning blocks and allowing code fences or surrounding text; this does not validate checklist completeness, item schemas, or correctness.

## Part 1: Score the fine-tuned model against your endpoint

Each request sends one contract and asks for the 17 verdicts. `chat_template_kwargs` carries
the same reasoning-off switch that training used, so the model answers directly instead of
deliberating; `max_tokens` of 700 is roughly triple what a complete answer needs.

In [ ]:
if RUN_ENDPOINT_EVAL:
    _t = time.time()
    _sample = E.ask_endpoint(
        EVAL_DOCS[0], runtime=runtime, endpoint_name=ENDPOINT_NAME,
        served_model_name=SERVED_MODEL_NAME, labels=labels)
    print(f"{time.time() - _t:.1f}s, {len(_sample)} chars\n")
    print(_sample[:400], "...")
else:
    print("RUN_ENDPOINT_EVAL is False, so the endpoint is not called")

In [ ]:
if RUN_ENDPOINT_EVAL:
    tuned_rows = E.score_all(
        EVAL_DOCS, runtime=runtime, endpoint_name=ENDPOINT_NAME,
        served_model_name=SERVED_MODEL_NAME, labels=labels, workers=WORKERS)
else:
    tuned_rows = E.load_rows("tuned_reference_rows.json", BASELINES)
    print(f"loaded {len(tuned_rows)} pre-computed answers from {BASELINES}")

tuned = E.summarise(tuned_rows)
for k, v in tuned.items():
    print(f"  {k:24s} {v:6.2f}")

## Optional: Streaming performance microbenchmark

`RUN_PERFORMANCE_BENCHMARK` defaults to `False` and is independent of quality scoring; enable it only against your running endpoint. Defaults are five short contracts, one separate serial warmup, one concurrent request, a 32 KiB serialized request cap, and 256 maximum output tokens. Inputs are not truncated; oversize requests fail before invocation, and the byte cap is not a token context limit.

Client elapsed seconds run from invocation to stream exhaustion; TTFT seconds stop at the first non-empty `delta.content`, not role, reasoning, or usage metadata. Output tokens/s is final `usage.completion_tokens / elapsed_s`, including prefill and network time, not decode-only throughput; absent final usage yields no token rate, never a character or chunk estimate. Per-request records retain usage and finish reason, and warmup and measured summaries each report sample count and linearly interpolated p50/p95.

The helper uses [SageMaker InvokeEndpointWithResponseStream](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_runtime_InvokeEndpointWithResponseStream.html) with `stream=true` and the shared chat-template flags; [vLLM 0.28.0 streaming source](https://github.com/vllm-project/vllm/blob/v0.28.0/vllm/entrypoints/openai/chat_completion/serving.py) implements `stream_options.include_usage` as a final usage chunk. Container-version support must still be checked when changing images; API or stream errors propagate without retries or fabricated metrics.

Short-input selection, warm caches, output caps, and small sample counts bias this microbenchmark; a `length` finish reason means capped output, not a quality result, and p95 from five samples is only exploratory.

In [ ]:
from inference_performance import benchmark, compute_cost

RUN_PERFORMANCE_BENCHMARK = False
BENCHMARK_SAMPLES = 5
BENCHMARK_WARMUPS = 1
BENCHMARK_CONCURRENCY = 1
BENCHMARK_MAX_REQUEST_BYTES = 32768
BENCHMARK_MAX_OUTPUT_TOKENS = 256
performance = None

if RUN_PERFORMANCE_BENCHMARK:
    candidates = [dict(model=SERVED_MODEL_NAME, messages=C.build_messages(d, labels),
                       chat_template_kwargs=C.CHAT_TEMPLATE_KWARGS,
                       temperature=0, max_tokens=BENCHMARK_MAX_OUTPUT_TOKENS)
                  for d in EVAL_DOCS]
    candidates.sort(key=lambda p: len(json.dumps(p).encode("utf-8")))
    if not 1 <= BENCHMARK_SAMPLES <= min(20, len(candidates)):
        raise ValueError("Choose 1-20 available benchmark contracts")
    performance = benchmark(runtime, ENDPOINT_NAME, candidates[:BENCHMARK_SAMPLES],
                            warmup_count=BENCHMARK_WARMUPS, concurrency=BENCHMARK_CONCURRENCY,
                            max_request_bytes=BENCHMARK_MAX_REQUEST_BYTES,
                            max_output_tokens=BENCHMARK_MAX_OUTPUT_TOKENS)
    print(json.dumps({k: v for k, v in performance.items()
                      if k not in ("samples", "warmup_samples")}, indent=2))
    for i, sample in enumerate(performance["samples"]):
        print(i, {k: v for k, v in sample.items() if k != "text"})
else:
    print("Streaming benchmark disabled; no benchmark requests sent")

### Optional: Serving compute allocation

Enter the verified SageMaker real-time hourly USD rate per instance for your region and deployed instance type from [SageMaker pricing](https://aws.amazon.com/sagemaker/ai/pricing/), not an EC2 rate; no price is assumed here. Measured wall time times fleet rate estimates microbenchmark compute allocation, not an actual per-request invoice; warmup is reported separately, and overlapping concurrent request durations must not be summed for fleet cost.

Supply serving lifetime seconds separately to estimate total provisioned compute, including idle time; this assumes a fixed instance count and excludes storage, transfer, taxes, and other charges, so it is not a billing reconciliation.

In [ ]:
SAGEMAKER_HOURLY_USD = None
PRICE_REGION = region
PRICE_INSTANCE_TYPE = None  # Enter the actual deployed type.
PRICE_SOURCE = None  # Record the pricing URL and lookup date.
SERVING_INSTANCE_COUNT = 1
SERVING_LIFETIME_SECONDS = None  # Include idle provisioned time.

if SAGEMAKER_HOURLY_USD is None:
    print("Set a verified regional SageMaker rate to estimate serving compute")
else:
    if PRICE_REGION != region or not PRICE_INSTANCE_TYPE or not PRICE_SOURCE:
        raise ValueError("Record the endpoint region, instance type, and price source")
    print(f"Rate basis: {PRICE_REGION}, {PRICE_INSTANCE_TYPE}, {PRICE_SOURCE}")
    if performance is not None:
        measured_usd = compute_cost(performance["measured_wall_s"], SAGEMAKER_HOURLY_USD, SERVING_INSTANCE_COUNT)
        warmup_usd = compute_cost(performance["warmup_wall_s"], SAGEMAKER_HOURLY_USD, SERVING_INSTANCE_COUNT)
        print(f"Microbenchmark compute allocation: ${measured_usd:.6f}; warmup: ${warmup_usd:.6f}")
        print(f"Allocation per measured request: ${measured_usd / len(performance['samples']):.6f}, not an invoice")
    if SERVING_LIFETIME_SECONDS is not None:
        lifetime_usd = compute_cost(SERVING_LIFETIME_SECONDS, SAGEMAKER_HOURLY_USD, SERVING_INSTANCE_COUNT)
        print(f"Separate serving lifetime compute estimate: ${lifetime_usd:.4f}")

## 4. Compare the other two models

<details>
<summary>How the saved untuned and frontier baselines were produced</summary>

## Part 2: The other two models

The untuned model and Sonnet 5 were scored with the same scorer on the same contracts, and
their results ship with the lab.

**The untuned 4B** takes about half an hour, and most of its answers do not parse. Its failure
has one shape: it emits well-formed JSON that never ends, listing dozens of span numbers as
evidence for every checklist item until it runs out of budget.

**Sonnet 5** is a live API call per contract, and it sees a slightly different prompt: the 4B is
prompted through its own chat template with reasoning disabled, while Sonnet gets the same
system and user turns as plain messages. Sonnet also does not accept a temperature setting, so
its answers are not greedy.

</details>

In [ ]:
base = E.load_precomputed("base", BASELINES)
frontier = E.load_precomputed("frontier", BASELINES)
ref = E.load_precomputed("tuned_reference", BASELINES)
E.compare_models(base, frontier, tuned, ref, MODEL_SLUG, live=RUN_ENDPOINT_EVAL)
E.plot_comparison(base, frontier, tuned)

## 5. LLM-as-a-Judge

The scoring above compares span **numbers**. That is exact and free, and unfair in one specific
way: if the model cites clause 47 and the annotator cited 48, and both say the same thing, the
scorer records a miss. With `evidence_f1` in the seventies, part of what is left is citations a
lawyer would accept and a string comparison will not.

A judge model reads the cited clause text and decides whether it justifies the verdict. It runs
as an Amazon Bedrock evaluation job with three custom metrics. The endpoint has already produced
the answers, so they go to Bedrock as **precomputed inference responses**: Bedrock judges, it
never loads the model.

> **IAM requirements.** The judge runs on Amazon Bedrock, so the execution role needs both:
>
> 1. a **trust relationship** allowing `bedrock.amazonaws.com` to assume it, and
> 2. an **identity-based policy** granting `bedrock:CreateEvaluationJob`, `GetEvaluationJob`,
>    `ListEvaluationJobs`, `StopEvaluationJob`, and `bedrock:InvokeModel` on the evaluator
>    model's inference profile.
>
> See [Bedrock evaluation permissions](https://docs.aws.amazon.com/bedrock/latest/userguide/judge-service-roles.html).

In [ ]:
EVALUATOR_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
JUDGE_N = 30
JUDGE_PREFIX = "contractnli-judge-eval"
JUDGE_OUTPUT = f"s3://{bucket_name}/{JUDGE_PREFIX}/output"
custom_metrics = E.build_judge_metrics()
METRIC_NAMES = [m["customMetricDefinition"]["name"] for m in custom_metrics]
print(f"{len(custom_metrics)} custom metrics, judged by {EVALUATOR_MODEL}")

### Prepare the rubric and submission

`E.build_judge_metrics()` retains all three original custom prompts and rating scales.
Inspect or modify `custom_metrics` before submitting a job.
The following function keeps the actual Bedrock submission payload visible.
It uses precomputed responses, so judging can run independently of the endpoint.

In [ ]:
def launch_judge(uri, name, n=JUDGE_N):
    """Create the Bedrock evaluation job and return its ARN."""
    bedrock = boto3.client("bedrock", region_name=region)
    job = bedrock.create_evaluation_job(
        jobName=f"contractnli-judge-{name}-{n}-{int(time.time())}",
        roleArn=role,
        applicationType="ModelEvaluation",
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "General",
                "dataset": {"name": "ContractNLI", "datasetLocation": {"s3Uri": uri}},
                "metricNames": METRIC_NAMES,
            }],
            "customMetricConfig": {
                "customMetrics": custom_metrics,
                "evaluatorModelConfig": {
                    "bedrockEvaluatorModels": [{"modelIdentifier": EVALUATOR_MODEL}]},
            },
        }},
        # precomputedInferenceSource is what lets a model Bedrock cannot host be judged here
        inferenceConfig={"models": [
            {"precomputedInferenceSource": {"inferenceSourceIdentifier": name}}]},
        outputDataConfig={"s3Uri": JUDGE_OUTPUT},
    )
    print(f"launched {name}: {job['jobArn']}")
    return job["jobArn"]

### Run it

`RUN_JUDGE_EVAL`, set in Setup, governs this part. Left `False` the notebook reads judge results
that ship with the lab. Set it `True` to run the judge yourself: two jobs, about 20 minutes, billed
per token against the evaluator model.

Both models are judged. The judge reads answers that already exist, either the ones Part 1 produced
or the ones that ship with the lab, so no endpoint is needed.

In [ ]:
JUDGE_FILE = f"{BASELINES}/judge_reference_metrics.json"
if RUN_JUDGE_EVAL:
    bedrock = boto3.client("bedrock", region_name=region)
    s3 = boto3.client("s3", region_name=region)
    base_rows = E.load_rows("base_rows.json", BASELINES)
    arns = {}
    for name, rows in (("base", base_rows), ("tuned", tuned_rows)):
        uri = E.build_judge_dataset(
            rows, name, s3=s3, bucket_name=bucket_name, documents=EVAL_DOCS,
            labels=labels, judge_prefix=JUDGE_PREFIX, n=JUDGE_N)
        arns[name] = launch_judge(uri, name)
    E.wait_for_judge(arns, bedrock=bedrock)
    judge_scores = {name: E.read_judge_scores(
        arn, bedrock=bedrock, s3=s3, bucket_name=bucket_name)
        for name, arn in arns.items()}
else:
    judge_scores = E.load_rows("judge_reference_metrics.json", BASELINES)["scores"]
    print(f"loaded pre-computed judge results from {JUDGE_FILE}")

In [ ]:
E.judge_table(judge_scores)
E.plot_judge_scores(judge_scores, JUDGE_N)

### Reading the judge

Two things about these numbers.

**They are normalised to 0-1.** A metric declared on a 0-3 scale comes back as 0, 0.333, 0.667 or
1.0, so `EvidenceSupportsVerdict` at 0.41 means 1.2 out of 3, not 41% of anything.

**Coverage is printed beside every mean.** Where it is low, or unequal between the two models, the
means describe different subsets of contracts and the gap between them is not a before and after
over the same work.

`ChecklistCompleteness` asks for well-formed JSON covering all 17 items; `json_valid` only checks whether the scorer extracted a JSON object, so it does not establish checklist completeness.

## Putting the two together

| | Statistical scorer | LLM-as-a-Judge |
| --- | --- | --- |
| Reproducible | identical every run | varies between runs |
| Cost | free | pays per token, on long prompts |
| Judges semantic citation validity | no, span numbers only | yes |
| Can be gamed | no | prompt injection is a real risk |
| Good for | regression gates, tracking a model over time | understanding why a model is wrong |

Use the statistical metrics as the gate and the judge to explain the residual. If the judge scores
`EvidenceSupportsVerdict` high while `evidence_f1` is low, the model is citing valid clauses the
annotator did not pick, which is a labelling artefact rather than a model failure.

## 6. Inspect error directions and clean up

In [ ]:
E.show_error_directions(tuned_rows, EVAL_DOCS)

### Takeaways

**Fine-tuning a 4B beat a frontier model on this task.** Not in general, on this task: a fixed
17-item checklist over a document type the model saw 423 examples of, where the answer has to
cite spans.

**Valid JSON is the first thing to read.** The untuned model's low scores are mostly not wrong
verdicts, they are answers that never finished. Without valid JSON the other three metrics
cannot be measured.

**Contradiction is the hardest class** for every model here, and the gap between the tuned model
and Sonnet is widest there. If this were going to production, that is the class to look at.

**Compare parseability and completeness separately.** `json_valid` checks object extraction, while the judge's `ChecklistCompleteness` also asks whether all items are present; neither establishes verdict accuracy.

## Clean up

> **Important:** a real-time endpoint bills per instance-hour for as long as it exists,
> whether or not you send it traffic.

Three resources, in dependency order: endpoint, then config, then model. The names are
rebuilt with the same helper notebook 3 used, since a fresh kernel shares no state.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

stem = f"{MODEL_SLUG}-contractnli"
for label, fn in [
    ("endpoint", lambda: Endpoint.get(ENDPOINT_NAME).delete()),
    ("endpoint config", lambda: EndpointConfig.get(E.rname(stem, "-sft-cfg")).delete()),
    ("model", lambda: Model.get(E.rname(stem, "-sft-m")).delete()),
]:
    try:
        fn()
        print(f"deleted {label}")
    except Exception as error:
        print(f"skip {label}: {error}")